# 06 — Resultados Globales y Comparación
**Ejecutar DESPUÉS de los notebooks 01–05.**
Agrega todos los resultados, genera las 16 gráficas de comparación, 4 gráficas resumen
y la matriz 4×4 del mejor modelo por combinación de ventanas.

> Pega aquí los dicts `results` de cada notebook (copia y pega las celdas de entrenamiento
> o re-ejecuta los notebooks antes de éste).

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.simplefilter('ignore')

from utils import INPUT_WINDOWS, OUTPUT_WINDOWS, build_results_df, plot_mae_matrix

## Paso 1 — Pegar resultados de todos los notebooks
Copia los dicts `results` y los valores de cada notebook a continuación.
El formato es: `{ (nombre_modelo, V_in, V_out): {'train':x, 'val':x, 'test':x, 'params':n} }`

In [ ]:
# ── PEGAR AQUÍ LOS results DE CADA NOTEBOOK ──────────────────
# Ejemplo:
#   results_01 = { ('naive',5,1): {'train':0.01,'val':0.01,'test':0.012,'params':0}, ... }
#   results_02 = { ('mlp_s',5,1): {'train':0.009, ...}, ... }
#   ...

# Combinar todos en un único dict
# results_all = {**results_01, **results_02, **results_03, **results_04, **results_05}

# PLACEHOLDER — reemplazar con los dicts reales tras ejecutar los notebooks
results_all = {}
print('Número de modelos cargados:', len(results_all))

In [ ]:
df_all = build_results_df(results_all)
print(f'Filas totales: {len(df_all)}')
df_all.head(10)

## Paso 2 — 16 gráficas: comparación de modelos por ventana

In [ ]:
if not df_all.empty:
    for V_in in INPUT_WINDOWS:
        for V_out in OUTPUT_WINDOWS:
            try:
                sub = df_all.xs((V_in, V_out), level=('V_in','V_out'),
                                drop_level=False)['test'].reset_index(level=[1,2], drop=True)
            except KeyError:
                continue

            fig, ax = plt.subplots(figsize=(8, 3))
            colors = ['tomato' if v == sub.min() else 'steelblue' for v in sub]
            sub.plot(kind='bar', ax=ax, color=colors, edgecolor='k')
            ax.set_title(f'MAE test — entrada={V_in}d, salida={V_out}d')
            ax.set_ylabel('MAE'); ax.set_xlabel('Modelo')
            ax.axhline(sub.min(), color='red', linestyle='--', alpha=0.5, label='Mejor')
            plt.xticks(rotation=30, ha='right'); plt.legend()
            plt.tight_layout(); plt.show()

## Paso 3 — 4 gráficas resumen por ventana de salida

In [ ]:
if not df_all.empty:
    for V_out in OUTPUT_WINDOWS:
        fig, ax = plt.subplots(figsize=(10, 4))
        modelos = df_all.index.get_level_values('modelo').unique()
        x = np.arange(len(INPUT_WINDOWS))
        width = 0.8 / len(modelos)

        for k, nombre in enumerate(modelos):
            maes = []
            for V_in in INPUT_WINDOWS:
                try:
                    v = df_all.loc[(nombre, V_in, V_out), 'test']
                except KeyError:
                    v = np.nan
                maes.append(v)
            ax.bar(x + k * width, maes, width, label=nombre)

        ax.set_xticks(x + width * (len(modelos)-1) / 2)
        ax.set_xticklabels([f'V_in={v}' for v in INPUT_WINDOWS])
        ax.set_ylabel('MAE test'); ax.set_title(f'Todos los modelos — V_out={V_out} días')
        ax.legend(loc='upper right', fontsize=8)
        plt.tight_layout(); plt.show()

## Paso 4 — Matriz 4×4: mejor MAE en test por combinación

In [ ]:
if not df_all.empty:
    mat = np.full((4, 4), np.nan)
    best_model = np.full((4, 4), '', dtype=object)

    for i, V_in in enumerate(INPUT_WINDOWS):
        for j, V_out in enumerate(OUTPUT_WINDOWS):
            try:
                sub = df_all.xs((V_in, V_out), level=('V_in','V_out'),
                                drop_level=False)['test']
                mat[i, j] = sub.min()
                best_model[i, j] = sub.idxmin()[0]   # nombre del modelo ganador
            except KeyError:
                pass

    df_mat = pd.DataFrame(mat, index=INPUT_WINDOWS, columns=OUTPUT_WINDOWS)
    plot_mae_matrix(df_mat, title='Mejor MAE test por combinación (todos los modelos)')

    print('\n── Modelo ganador por combinación ──')
    df_best = pd.DataFrame(best_model, index=INPUT_WINDOWS, columns=OUTPUT_WINDOWS)
    df_best.index.name = 'V_in'; df_best.columns.name = 'V_out'
    print(df_best.to_string())

## Paso 5 — Tabla completa con número de parámetros

In [ ]:
if not df_all.empty:
    # Resaltar el mejor MAE test en cada (V_in, V_out)
    def highlight_min(s):
        return ['background-color: lightgreen' if v == s.min() else '' for v in s]

    display(df_all.style.apply(highlight_min, subset=['test']))

## Reflexión — ¿Qué arquitecturas funcionan mejor y en qué ventanas?
Completar tras analizar los resultados.

- **Ventanas cortas (V_out=1,5)**: ...
- **Ventanas largas (V_out=30,90)**: ...
- **Mejor tipo de modelo global**: ...